# Tabela única dos PDFs Carros na Web

Este notebook processa todos os arquivos PDF dentro de `carros_benchmark` com `use_llm_fallback=False` e consolida o resultado em uma tabela wide:

- cada linha é um atributo veicular
- cada coluna é um veículo analisado

In [1]:
from pathlib import Path
import re

import pandas as pd

from carros_na_web_parser import parse_pdf

In [2]:
base_dir = Path('carros_benchmark')
pdf_paths = sorted(base_dir.glob('*.pdf'))
pdf_paths

[WindowsPath('carros_benchmark/Carros na Web _ BYD Song Plus GS 1.5 2027 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Jeep Compass Blackhawk 2.0 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Renault Boreal Iconic 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Toyota Corolla Cross XRX 2.0 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Volkswagen Taos Comfortline 1.4 TSi 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf')]

In [3]:
def vehicle_name_from_pdf_path(pdf_path: Path) -> str:
    name = pdf_path.stem
    name = re.sub(r'^Carros na Web\s*_\s*', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s*_\s*Ficha Técnica.*$', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s+', ' ', name).strip()
    return name


vehicle_name_from_pdf_path(pdf_paths[0]) if pdf_paths else None

'BYD Song Plus GS 1.5 2027'

In [4]:
frames = {}

def collapse_duplicate_attributes(df: pd.DataFrame) -> pd.Series:
    cleaned = df.copy()
    cleaned['Atributo veicular'] = cleaned['Atributo veicular'].astype(str).str.strip()
    cleaned['Valor'] = cleaned['Valor'].astype(str).str.strip()
    return cleaned.groupby('Atributo veicular', sort=False)['Valor'].agg(
        lambda values: next((value for value in values if value and value != 'nan'), '')
    )

for pdf_path in pdf_paths:
    vehicle_name = vehicle_name_from_pdf_path(pdf_path)
    df = parse_pdf(pdf_path, use_llm_fallback=False)
    frames[vehicle_name] = df
    print(f'{vehicle_name}: {len(df)} linhas')

frames.keys()

BYD Song Plus GS 1.5 2027: 173 linhas
Jeep Compass Blackhawk 2.0 2026: 170 linhas
Renault Boreal Iconic 1.3 2026: 166 linhas
Toyota Corolla Cross XRX 2.0 2026: 158 linhas
Volkswagen Taos Comfortline 1.4 TSi 2026: 150 linhas


dict_keys(['BYD Song Plus GS 1.5 2027', 'Jeep Compass Blackhawk 2.0 2026', 'Renault Boreal Iconic 1.3 2026', 'Toyota Corolla Cross XRX 2.0 2026', 'Volkswagen Taos Comfortline 1.4 TSi 2026'])

In [5]:
frames

{'BYD Song Plus GS 1.5 2027':                                 Atributo veicular     Valor
 0                                             Ano      2027
 1                                           Preço    R$ 249
 2                                            IPVA   R$ 10.0
 3                                          Seguro   R$ 6.75
 4                                        Revisões   R$ 6.38
 ..                                            ...       ...
 168                  Sistema de som premium Infin  Presente
 169                                     Subwoofer  Presente
 170                               Head-up display  Presente
 171                   Leitor de cartão de memória  Presente
 172  Informações do veículo através de aplicativo  Presente
 
 [173 rows x 2 columns],
 'Jeep Compass Blackhawk 2.0 2026':                         Atributo veicular     Valor
 0                                     Ano      2026
 1                                   Preço  R$ 224.6
 2                

In [6]:
wide = pd.concat(
    {vehicle: collapse_duplicate_attributes(df) for vehicle, df in frames.items()},
    axis=1,
).reset_index()

wide = wide.rename(columns={'index': 'Atributo veicular'})
wide

,Atributo veicular,BYD Song Plus GS 1.5 2027,Jeep Compass Blackhawk 2.0 2026,Renault Boreal Iconic 1.3 2026,Toyota Corolla Cross XRX 2.0 2026,Volkswagen Taos Comfortline 1.4 TSi 2026
0,Ano,2027,2026,2026,2026,2026
1,Preço,R$ 249,R$ 224.6,R$ 214.990,R$ 194.700,R$ 199.990
2,IPVA,R$ 10.0,R$ 8.984,R$ 8.600,R$ 7.788,R$ 8.000
3,Seguro,R$ 6.75,R$ 7.187,R$ 5.805,R$ 6.230,R$ 5.400
4,Revisões,R$ 6.38,R$ 4.127,R$ 6.316 até 60.000,R$ 4.782 até 50.000,R$ 3.331 até 60.000
...,...,...,...,...,...,...
292,ISOFIX para fixação de cadei,NaN,NaN,NaN,NaN,Presente
293,Frenagem automática de eme,NaN,NaN,NaN,NaN,Presente
294,Vetorização de torque,NaN,NaN,NaN,NaN,Presente
295,Banco do motorista com ajust,NaN,NaN,NaN,NaN,Presente


In [7]:
# Visão rápida do resultado final
print(f'Veículos processados: {len(frames)}')
print(f'Atributos únicos: {len(wide)}')
wide.head(20).to_string(index=False)

Veículos processados: 5
Atributos únicos: 297


'Atributo veicular BYD Song Plus GS 1.5 2027 Jeep Compass Blackhawk 2.0 2026 Renault Boreal Iconic 1.3 2026 Toyota Corolla Cross XRX 2.0 2026 Volkswagen Taos Comfortline 1.4 TSi 2026\n              Ano                      2027                            2026                           2026                              2026                                     2026\n            Preço                    R$ 249                        R$ 224.6                     R$ 214.990                        R$ 194.700                               R$ 199.990\n             IPVA                   R$ 10.0                        R$ 8.984                       R$ 8.600                          R$ 7.788                                 R$ 8.000\n           Seguro                   R$ 6.75                        R$ 7.187                       R$ 5.805                          R$ 6.230                                 R$ 5.400\n         Revisões                   R$ 6.38                        R$ 4.127         

In [9]:
# Se quiser persistir o resultado, descomente uma das linhas abaixo.
wide.to_csv('carros_benchmark_wide.csv', index=False, encoding='utf-8-sig')
wide.to_excel('carros_benchmark_wide.xlsx', index=False)

wide

,Atributo veicular,BYD Song Plus GS 1.5 2027,Jeep Compass Blackhawk 2.0 2026,Renault Boreal Iconic 1.3 2026,Toyota Corolla Cross XRX 2.0 2026,Volkswagen Taos Comfortline 1.4 TSi 2026
0,Ano,2027,2026,2026,2026,2026
1,Preço,R$ 249,R$ 224.6,R$ 214.990,R$ 194.700,R$ 199.990
2,IPVA,R$ 10.0,R$ 8.984,R$ 8.600,R$ 7.788,R$ 8.000
3,Seguro,R$ 6.75,R$ 7.187,R$ 5.805,R$ 6.230,R$ 5.400
4,Revisões,R$ 6.38,R$ 4.127,R$ 6.316 até 60.000,R$ 4.782 até 50.000,R$ 3.331 até 60.000
...,...,...,...,...,...,...
292,ISOFIX para fixação de cadei,NaN,NaN,NaN,NaN,Presente
293,Frenagem automática de eme,NaN,NaN,NaN,NaN,Presente
294,Vetorização de torque,NaN,NaN,NaN,NaN,Presente
295,Banco do motorista com ajust,NaN,NaN,NaN,NaN,Presente
